In [ ]:
%%file producer.py

from kafka import KafkaProducer
import json, random, time
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='broker:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

miasta = ["Warszawa", "Kraków", "Gdańsk", "Wrocław"]
kategorie = ["elektronika","odzież","żywność","książki"]
#Każda transakcja to JSON z polami: - tx_id (np. “TX0001”) - user_id (losowy z “u01” do “u20”) - amount (losowy float 5.0–5000.0) - store (losowy z: #“Warszawa”, “Kraków”, “Gdańsk”, “Wrocław”) - category (losowy z: “elektronika”, “odzież”, “żywność”, “książki”) - timestamp (aktualny czas ISO)

def generate_transaction():
    slownik = {
        "tx_id": f"TX{random.randint(1, 9999):04d}",
        "user_id": f"u{random.randint(1, 20):02d}",
        "amount": round(random.uniform(5.0, 5000.0), 2),
        "store": random.choice(miasta),
        "category": random.choice(kategorie),
        "timestamp": datetime.now().isoformat()
    }

    return slownik

# TWÓJ KOD
# Pętla: generuj transakcję, wyślij do tematu 'transactions', wypisz, sleep 1s

while(True):
    transaction = generate_transaction()
    producer.send('transactions', value=transaction)
    producer.flush()
    print(transaction)
    time.sleep(1)

In [3]:
%%file consumer_filter.py
from kafka import KafkaConsumer
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='enrich-group'
)


for message in consumer:
    transaction = message.value
    if transaction["amount"] > 3000:
        print(
            f"ALERT: {transaction['tx_id']} | "
            f"{transaction['amount']} PLN | "
            f"{transaction['store']} | "
            f"{transaction['category']}"
        )
    
# TWÓJ KOD
# Dla każdej wiadomości: sprawdź czy amount > 3000, jeśli tak — wypisz ALERT
# Format: ALERT: TX0042 | 2345.67 PLN | Warszawa | elektronika

Overwriting consumer_filter.py


In [4]:
%%file consumer_enrich.py
from kafka import KafkaConsumer
import json

# TWÓJ KOD
# Czytaj z 'transactions' (użyj INNEGO group_id!)
# Dodaj pole risk_level na podstawie amount
# Wypisz wzbogaconą transakcję

#Napisz konsumenta, który do słownika eventu dodaje pole risk_level: 
#- amount > 3000 → “HIGH” - amount > 1000 → “MEDIUM” - pozostałe → “LOW”

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    group_id='enrich-group',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    group_id='enrich-group'
)

for message in consumer:
    transaction = message.value

    if transaction["amount"] > 3000:
        transaction["risk_level"] = "HIGH"
    elif transaction["amount"] > 1000:
        transaction["risk_level"] = "MEDIUM"
    else:
        transaction["risk_level"] = "LOW"

    print(transaction)
    

Overwriting consumer_enrich.py


In [1]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

store_counts = Counter()
total_amount = {}
msg_count = 0

for message in consumer:
    transaction = message.value
    store = transaction['store']
    
    store_counts[store] += 1
    
    if store not in total_amount:
        total_amount[store] = 0
    total_amount[store] += transaction['amount']
    
    msg_count += 1
    
    if msg_count % 10 == 0:
        print("Sklep | Liczba | Suma | Średnia")
        for m in store_counts:
            print(
                f"{m} | {store_counts[m]} | {total_amount[m]} | {total_amount[m]/store_counts[m]}"
            )

Overwriting consumer_count.py


In [2]:
%%file consumer_stats.py
from kafka import KafkaConsumer
from collections import defaultdict, Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

cat_counts = Counter()
total_amount = {}
minn = {}
maxx = {}
msg_count = 0

for message in consumer:
    transaction = message.value
    cat = transaction['category']
    amount = transaction['amount']

    cat_counts[cat] += 1

    if cat not in total_amount:
        total_amount[cat] = 0
    total_amount[cat] += amount

    if cat not in minn:
        minn[cat] = amount
        maxx[cat] = amount
    else:
        if amount < minn[cat]:
            minn[cat] = amount
        if amount > maxx[cat]:
            maxx[cat] = amount

    msg_count += 1

    if msg_count % 10 == 0:
        print("Kategoria | Liczba | Suma | Min | Max")
        for m in cat_counts:
            print(
                f"{m} | {cat_counts[m]} | {total_amount[m]} | {minn[m]} | {maxx[m]}"
            )

Writing consumer_stats.py


In [ ]:
#Co się stanie, jeśli uruchomisz consumer_filter.py po zakończeniu producenta?
#Będzie nasłuchiwać dalej
#Co się stanie, jeśli dwóch konsumentów ma TĘ SAMĄ group_id?
#Jaka jest różnica między przetwarzaniem bezstanowym a stanowym?

In [6]:
%%file consumer_count.py
from kafka import KafkaConsumer
from collections import Counter
import json

consumer = KafkaConsumer(
    'transactions',
    bootstrap_servers='broker:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

client_counts = Counter()
msg_count = 0

for message in consumer:
    transaction = message.value
    user_id = transaction['user_id']

    client_counts[user_id] += 1
    msg_count += 1

    if msg_count % 60 == 0:
        for user in client_counts:
            if client_counts[user] > 3:
                print(f"ALERT: klient {user} wykonał {client_counts[user]} transakcje w ostatnich 60 sekundach")
        client_counts.clear() 
        print("***************************************")

Overwriting consumer_count.py
